# F_ATTN fixed-point precision sweep

Bit-exact model of the fixed-point F_ATTN datapath (`01_RTL/F_ATTN.v`) and a GPU sweep over all
combinations of the 9 width parameters. Every configuration is run on 10,000 patterns
(`gen_pattern.py` seeds 1 ~ 10) and compared with the golden (double precision, same as
`ref_model` in PATTERN). The final table lists the configurations whose max error meets `TOL`.

| parameter | meaning (fraction bits unless noted) | hardware it sizes |
|---|---|---|
| Nc | bits of c = log2(e) / sqare_root_2 | Q converter constant multiplier |
| Fq | Q' = Q * c | Q buffer, S1 multipliers |
| Fk | K | K buffer, S1 multipliers |
| Fv | V | V buffer, S3 multipliers |
| Fw | W | W buffer, dp4 multipliers |
| Ft | t = Q' . K (exponent of 2) | S2 registers, pow2 input |
| Pw | weight 2^(t - r), D | pow2 output, S3 multipliers, D |
| Fm | M | M registers, divider numerator |
| Fh | H = M / D (\|H\| saturated to Fh - 1 bits) | SRT steps, H registers, dp4 |

Fixed design choices (not swept): \|H\| + 1/2 LSB in the linear layer, t and w * V round half up,
pow2 = table + quadratic (ATTN_POW2), fp32 output by one's complement and truncation.

Run order: cell 1 ~ 9 (cell 9 only needs cells 1 ~ 5). Cells 2 / 3 write `fxp_model.py` / `fxp_gpu.py` (Windows multiprocessing
and numba need the code in importable files). Needs a CUDA GPU and `numba-cuda`.

In [1]:
# Cell 1: settings
import os, sys, time, importlib
import numpy as np
import pandas as pd
from multiprocessing import Pool

HERE = os.getcwd()
GEN = os.path.join(HERE, '..', '00_TESTBED', 'gen_pattern.py')   # pattern generator
PAT_DIR = os.path.join(HERE, 'pattern')                           # seed_1 ~ 10 + data.npz (golden cache)
OUT_DIR = os.path.join(HERE, 'results')
SEEDS = range(1, 11)                                              # 10 x 1000 patterns
TOL = 1e-6                                                        # demo tolerance

# widths of the current hardware (01_RTL/F_ATTN.v parameters): checked GPU vs CPU in cell 5
# and marked in the tables
CURRENT_HW = dict(Nc=19, Fq=25, Fk=16, Fv=22, Fw=22, Ft=20, Pw=23, Fm=24, Fh=22)

# sweep range of every width (inclusive). Limits: the RTL constant tables cover NC 14 ~ 24,
# FT 15 ~ 22, PW 20 ~ 27 and need FT >= PW - 9 (other combinations are skipped); the GPU kernel
# needs every intermediate < 2^63 (largest: Q'.K < 2^(Fq + Fk + 9))
RANGES = dict(Nc=(15, 21), Fq=(23, 28), Fk=(14, 18), Fv=(19, 24), Fw=(19, 25),
              Ft=(16, 22), Pw=(20, 26), Fm=(21, 27), Fh=(19, 25))

# candidate ranking (cell 8): stricter error limit for a safety margin, and the area (x 1000 cell
# area units) one bit of each width costs. Calibrated with DC at 10.5 ns on 11 configs: Fh and Nc
# were measured directly, Fq + Pw, Fv - Fw and Ft - Fm only as sums / differences (the split is
# assumed), Fk not at all. The cost depends on the clock (Fh is high at 10.5 ns because the SRT
# divider is on the critical path)
MARGIN_TOL = 6e-7
AREA_PER_BIT = dict(Nc=0.1, Fq=7.0, Fk=6.6, Fv=4.0, Fw=3.0, Ft=0.5, Pw=8.5, Fm=1.0, Fh=7.0)

# ATTN_POW2 internal precision sweep (cell 9), the 9 widths fixed at CURRENT_HW
#   HB table index bits, TB table fraction bits, A1B / A2B a1 / a2 fraction bits (A2B = 0: no x^2),
#   XQ fraction bits kept in xq, EB fraction bits of e = 2^x - 1, ST low bits of T dropped in T * e
#   (current RTL with Pw = 23: HB 6, TB 23, A1B 18, XQ 14, A2B 7, EB 24, ST 7)
POW2_RANGES = dict(HB=range(4, 9), TB=range(19, 24), A1B=range(13, 20), XQ=range(10, 17),
                   A2B=[0, 3, 4, 5, 6, 7, 8], EB=range(20, 26), ST=range(5, 12))
# pow2 cost proxy until DC calibration: multiplier array bits + ROM_W * table bits
ROM_W = 0.25

os.makedirs(OUT_DIR, exist_ok=True)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)
print('configs in the ranges:', np.prod([hi - lo + 1 for lo, hi in RANGES.values()]))

configs in the ranges: 21176820


In [2]:
%%writefile fxp_model.py
# F_ATTN fixed-point model (CPU): parametric, bit-exact with 01_RTL/F_ATTN.v at BASE
# (the reference for the GPU sweep kernel in fxp_gpu.py)
import struct, math, os, sys, pickle, subprocess
import numpy as np

SQ2 = struct.unpack('>f', bytes.fromhex('3FB504F3'))[0]
C = math.log2(math.e) / SQ2
KEYS = ['Nc', 'Fq', 'Fk', 'Fv', 'Fw', 'Ft', 'Pw', 'Fm', 'Fh']
# widths of 01_RTL (r08); hhalf / tround / pvround / pow2 are design options, not swept
BASE = dict(Nc=19, Fq=25, Fk=16, Fv=22, Fw=22, Ft=20, Pw=23, Fm=24, Fh=22,
            pow2='hw', hhalf=1, pvround=1, tround=1)

#=============================================================
#                      Bit-exact Model
#=============================================================
def fpb(h):
    b = int(h, 16) if isinstance(h, str) else int(h)
    s = b >> 31; e = (b >> 23) & 0xFF
    return s, e, (0 if e == 0 else (b & 0x7FFFFF) | 0x800000)

def shr(x, n): return (x >> n) if n >= 0 else (x << -n)

# ATTN_POW2 internal precision (fraction bits unless noted), defaults = the RTL derivation from Pw:
#   HB  table index bits (2^HB entries)        TB  table entry fraction bits (<= Pw)
#   A1B a1 = round(a1 * 2^A1B)                 XQ  fraction bits kept in xq (x squared)
#   A2B a2 = round(a2 * 2^A2B), 0 = no x^2     EB  fraction bits of e = 2^x - 1
#   ST  low bits of T dropped in T * e
P2KEYS = ['HB', 'TB', 'A1B', 'XQ', 'A2B', 'EB', 'ST']

def pow2_defaults(Pw):
    return dict(HB=6, TB=Pw, A1B=Pw - 5, XQ=Pw - 9, A2B=Pw - 16, EB=Pw + 1, ST=Pw - 16)

_fit = {}
def pow2_fit(HB):
    # a1 / a2 of 2^x - 1 ~ a1 x + a2 x^2, least squares on [0, 2^-HB)
    if HB not in _fit:
        xs = np.linspace(0, 2.0 ** -HB, 4001)
        _fit[HB] = np.linalg.lstsq(np.vstack([xs, xs ** 2]).T, 2.0 ** xs - 1, rcond=None)[0]
    return _fit[HB]

_exact = {}
def pow2_exact(Ft, Pw):
    # sample points f and exact 2^f * 2^Pw used to center the truncation error
    if (Ft, Pw) not in _exact:
        f = list(range(0, 1 << Ft, max(1, (1 << Ft) // 10800)))
        _exact[(Ft, Pw)] = (np.array(f, dtype=np.int64), np.array([2.0 ** (x / (1 << Ft)) * (1 << Pw) for x in f]))
    return _exact[(Ft, Pw)]

def shr_np(x, n): return (x >> n) if n >= 0 else (x << -n)

def pow2_table(HB, TB, Pw):
    # 2^(i / 2^HB) rounded to TB fraction bits, aligned to Pw fraction bits
    return [round(2.0 ** (i / (1 << HB)) * (1 << TB)) << (Pw - TB) for i in range(1 << HB)]

def pow2_job(a):
    # (Ft, Pw, q) -> (A1, A2, bias), for a process pool
    Ft, Pw, q = a
    return tuple(pow2_params(Ft, Pw, **q)[1:4])

def pow2_params(Ft, Pw, **kw):
    # table (aligned to Pw fraction bits), a1, a2, bias and the full parameter set
    q = pow2_defaults(Pw); q.update(kw)
    HB, TB, A1B, XQ, A2B, EB, ST = (q[k] for k in P2KEYS)
    T = pow2_table(HB, TB, Pw)
    a1, a2 = pow2_fit(HB)
    A1 = round(a1 * 2 ** A1B)
    A2 = round(a2 * 2 ** A2B) if A2B > 0 else 0
    fs, ex = pow2_exact(Ft, Pw)
    x = fs & ((1 << (Ft - HB)) - 1); t = np.array(T, dtype=np.int64)[fs >> (Ft - HB)]
    xq = shr_np(x, Ft - XQ)
    e = shr_np(x * A1, A1B + Ft - EB) + shr_np(xq * xq * A2, 2 * XQ + A2B - EB)
    d = (t + shr_np((t >> ST) * e, EB - ST)).astype(np.float64) - ex
    bias = int(round(-(d.max() + d.min()) / 2))
    return T, A1, A2, bias, q

def pow2_consts(Ft, Pw):
    # default ATTN_POW2 (64-entry table): table, a1, a2, bias
    return pow2_params(Ft, Pw)[:4]

def pow2_raw(T, A1, A2, Ft, Pw, q):
    HB, TB, A1B, XQ, A2B, EB, ST = (q[k] for k in P2KEYS)
    xm = (1 << (Ft - HB)) - 1
    def nb(fr):
        x = fr & xm; fh = fr >> (Ft - HB)
        lin = shr(x * A1, A1B + Ft - EB)
        xq = shr(x, Ft - XQ)
        quad = shr(xq * xq * A2, 2 * XQ + A2B - EB)
        return T[fh] + shr((T[fh] >> ST) * (lin + quad), EB - ST)
    return nb

def make_pow2(p):
    Ft, Pw = p['Ft'], p['Pw']
    if p['pow2'] == 'ideal':
        return lambda fr: int(math.floor(2.0 ** (fr / (1 << Ft)) * (1 << Pw) + 0.5))
    T, A1, A2, bias, q = pow2_params(Ft, Pw, **{k: p[k] for k in P2KEYS if k in p})
    nb = pow2_raw(T, A1, A2, Ft, Pw, q)
    return lambda fr: nb(fr) + bias

def to_fp32(acc, F):
    # one's complement |acc| (|acc| - 1 when negative) and a truncated mantissa, as in the RTL
    s = 1 if acc < 0 else 0
    mag = (~acc) if s else acc
    if mag == 0: return 0.0
    pos = mag.bit_length() - 1
    mant = (mag >> (pos - 23)) if pos >= 23 else (mag << (23 - pos))
    return (-1.0 if s else 1.0) * mant * 2.0 ** (pos - 23 - F)

def model(pat, p, pow2):
    # pat = (q, k, v, w): fp32 words (hex strings or ints), 64 / 64 / 64 / 16
    q, k, v, w = pat
    Nc, Fq, Fk, Fv, Fw, Ft, Pw, Fm, Fh = (p[x] for x in KEYS)
    cint = round(C * (1 << Nc))
    def cq(h):                                          # Q' = Q * c, round half up
        s, e, m = fpb(h)
        if m == 0 or e > 126: return 0
        x = shr(m * cint, 150 + Nc - Fq - 1 - e); x = (x + 1) >> 1
        return -x if s else x
    def cf(h, F, emax):                                 # K / V / W, round half up
        s, e, m = fpb(h)
        if m == 0 or e > emax: return 0
        x = shr(m, 149 - F - e); x = (x + 1) >> 1
        return -x if s else x
    Q = [cq(x) for x in q]; K = [cf(x, Fk, 134) for x in k]; V = [cf(x, Fv, 126) for x in v]; W = [cf(x, Fw, 126) for x in w]
    st = Fq + Fk - Ft; tr = (1 << (st - 1)) if p['tround'] else 0
    sp = Pw + Fv - Fm; pr = (1 << (sp - 1)) if p['pvround'] else 0
    sh_h = Fh - Fm + Pw; hmax = (1 << (Fh - 1)) - 1
    fmask = (1 << Ft) - 1
    H = [[0] * 4 for _ in range(16)]
    for h in range(2):
        for i in range(16):
            qa, qb = Q[4 * i + 2 * h], Q[4 * i + 2 * h + 1]
            for j in range(8):                          # key tiles in the RTL order
                ka = 4 * (2 * j) + 2 * h; kb = 4 * (2 * j + 1) + 2 * h
                # S1: t = Q' . K
                t0 = (qa * K[ka] + qb * K[ka + 1] + tr) >> st
                t1 = (qa * K[kb] + qb * K[kb + 1] + tr) >> st
                # S2: integer running max r, weight = 2^frac(t) >> (r - floor t)
                f0, f1 = t0 >> Ft, t1 >> Ft; r0, r1 = t0 & fmask, t1 & fmask
                c0 = f0 + (1 if r0 else 0); c1 = f1 + (1 if r1 else 0)
                rn = max(c0, c1) if j == 0 else max(r, c0, c1)
                w0 = pow2(r0) >> min(rn - f0, 31); w1 = pow2(r1) >> min(rn - f1, 31)
                # S3: D / M accumulation with the shift correction
                pv0 = (w0 * V[ka] + w1 * V[kb] + pr) >> sp
                pv1 = (w0 * V[ka + 1] + w1 * V[kb + 1] + pr) >> sp
                if j == 0:
                    D = w0 + w1; M0 = pv0; M1 = pv1
                else:
                    dl = min(rn - r, 31)
                    D = (D >> dl) + w0 + w1; M0 = (M0 >> dl) + pv0; M1 = (M1 >> dl) + pv1
                r = rn
            # divider: |H| = floor(|M| / D), saturated; + 1/2 LSB in the linear layer
            for d, M in ((0, M0), (1, M1)):
                hq = min(shr(abs(M), -sh_h) // D, hmax)
                if p['hhalf']: hq = 2 * hq + 1
                H[i][2 * h + d] = -hq if M < 0 else hq
    # linear layer + fp32
    F = Fh + (1 if p['hhalf'] else 0) + Fw
    return [to_fp32(sum(H[t_][c] * W[4 * o + c] for c in range(4)), F) for t_ in range(16) for o in range(4)]

#=============================================================
#                    Golden and Patterns
#=============================================================
def golden(pat):
    # standard (non-tiled) attention in double precision, as ref_model in PATTERN
    q, k, v, w = pat
    f = lambda h: struct.unpack('>f', struct.pack('>I', int(h, 16) if isinstance(h, str) else int(h)))[0]
    Q = [[f(q[4 * i + d]) for d in range(4)] for i in range(16)]; K = [[f(k[4 * i + d]) for d in range(4)] for i in range(16)]
    V = [[f(v[4 * i + d]) for d in range(4)] for i in range(16)]; W = [[f(w[4 * i + d]) for d in range(4)] for i in range(4)]
    head = [[0] * 4 for _ in range(16)]
    for h in range(2):
        for i in range(16):
            s = [(Q[i][2 * h] * K[j][2 * h] + Q[i][2 * h + 1] * K[j][2 * h + 1]) / SQ2 for j in range(16)]
            m = max(s); e = [math.exp(x - m) for x in s]; sm = sum(e)
            for d in range(2): head[i][2 * h + d] = sum(e[j] * V[j][2 * h + d] for j in range(16)) / sm
    return [sum(head[t][c] * W[o][c] for c in range(4)) for t in range(16) for o in range(4)]

def load(path):
    # input.txt format: pattern count, then per cycle "Q K V [W]" (W only in the first 16 cycles)
    t = open(path).read().split(); n = int(t[0]); it = iter(t[1:]); P = []
    for _ in range(n):
        q, k, v, w = [], [], [], []
        for c in range(64):
            q.append(next(it)); k.append(next(it)); v.append(next(it))
            if c < 16: w.append(next(it))
        P.append((q, k, v, w))
    return P

def make_patterns(pat_dir, gen, seeds):
    os.makedirs(pat_dir, exist_ok=True)
    files = []
    for s in seeds:
        fn = os.path.join(pat_dir, 'seed_%d.txt' % s)
        if not os.path.exists(fn):
            subprocess.run([sys.executable, gen, '--pat', '1000', '--seed', str(s), '-o', fn], check=True, stdout=subprocess.DEVNULL)
        files.append(fn)
    return files

def _golden_chunk(pats): return [golden(x) for x in pats]

def load_data(files, cache, pool=None):
    # patterns as uint32 arrays (Q, K, V: P x 64, W: P x 16) + golden (P x 64), cached in an .npz
    if os.path.exists(cache):
        d = np.load(cache)
        return d['Q'], d['K'], d['V'], d['W'], d['G']
    P = []
    for fn in files: P += load(fn)
    if pool is None: G = [golden(x) for x in P]
    else: G = [g for part in pool.map(_golden_chunk, [P[i:i + 250] for i in range(0, len(P), 250)]) for g in part]
    to = lambda idx, n: np.array([[int(x, 16) for x in pat[idx]] for pat in P], dtype=np.uint32).reshape(len(P), n)
    Q, K, V, W = to(0, 64), to(1, 64), to(2, 64), to(3, 16)
    G = np.array(G, dtype=np.float64)
    np.savez(cache, Q=Q, K=K, V=V, W=W, G=G)
    return Q, K, V, W, G

def as_pat(Q, K, V, W, i): return ([int(x) for x in Q[i]], [int(x) for x in K[i]], [int(x) for x in V[i]], [int(x) for x in W[i]])

def _err_chunk(args):
    # per-pattern max |out - golden| for one config over patterns lo ~ hi (CPU reference)
    p, Q, K, V, W, G = args
    pw = make_pow2(p)
    return [max(abs(a - b) for a, b in zip(model(as_pat(Q, K, V, W, i), p, pw), G[i])) for i in range(len(G))]

def cpu_errors(pool, p, Q, K, V, W, G, chunk=200):
    n = len(G)
    jobs = [(p, Q[i:i + chunk], K[i:i + chunk], V[i:i + chunk], W[i:i + chunk], G[i:i + chunk]) for i in range(0, n, chunk)]
    return np.array([e for part in pool.map(_err_chunk, jobs) for e in part])

Overwriting fxp_model.py


In [3]:
%%writefile fxp_gpu.py
# F_ATTN fixed-point model on the GPU (numba-cuda): the same integer datapath as fxp_model.model,
# one thread = one (config, pattern), a block reduces 128 patterns of one config
# all intermediates fit in int64 for the sweep ranges (largest: Q'.K sum < 2^(Fq + Fk + 9))
import math, os, time
import numpy as np
from numba import cuda, int64, float64
from numba.cuda import libdevice
import fxp_model as fm

TPB = 128                                           # patterns per block

#=============================================================
#                      Device Functions
#=============================================================
@cuda.jit(device=True, inline=True)
def shr(x, n):
    if n >= 0:
        if n > 62:
            return x >> 63                          # 0 or -1, as a Python shift by a large n
        return x >> n
    return x << (-n)

@cuda.jit(device=True, inline=True)
def conv_q(b, cint, Nc, Fq):
    e = (b >> 23) & 0xFF
    if e == 0 or e > 126:
        return int64(0)
    m = (b & 0x7FFFFF) | 0x800000
    x = shr(m * cint, 150 + Nc - Fq - 1 - e)
    x = (x + 1) >> 1
    if (b >> 31) & 1:
        return -x
    return x

@cuda.jit(device=True, inline=True)
def conv_f(b, F, emax):
    e = (b >> 23) & 0xFF
    if e == 0 or e > emax:
        return int64(0)
    m = (b & 0x7FFFFF) | 0x800000
    x = shr(m, 149 - F - e)
    x = (x + 1) >> 1
    if (b >> 31) & 1:
        return -x
    return x

@cuda.jit(device=True, inline=True)
def pow2(fr, Ft, T, ti, HB, XQ, EB, ST, A1B, A2B, A1, A2, bias):
    # ATTN_POW2 with its internal precision (see fxp_model.P2KEYS), table row ti
    x = fr & ((int64(1) << (Ft - HB)) - 1)
    fh = fr >> (Ft - HB)
    lin = shr(x * A1, A1B + Ft - EB)
    xq = shr(x, Ft - XQ)
    quad = shr(xq * xq * A2, 2 * XQ + A2B - EB)
    t = T[ti, fh]
    return t + shr((t >> ST) * (lin + quad), EB - ST) + bias

@cuda.jit(device=True, inline=True)
def hval(M, D, shh, hmax):
    a = M
    if a < 0:
        a = -a
    hq = shr(a, -shh) // D
    if hq > hmax:
        hq = hmax
    hq = 2 * hq + 1                                 # |H| + 1/2 LSB
    if M < 0:
        return -hq
    return hq

@cuda.jit(device=True, inline=True)
def to_fp(acc, F):
    neg = acc < 0
    mag = acc
    if neg:
        mag = ~acc
    if mag == 0:
        return 0.0
    pos = 63 - libdevice.clzll(mag)
    if pos >= 23:
        mant = mag >> (pos - 23)
    else:
        mant = mag << (23 - pos)
    v = libdevice.ldexp(float64(mant), pos - 23 - F)
    if neg:
        return -v
    return v

#=============================================================
#                          Kernel
#=============================================================
@cuda.jit
def sweep_kernel(Qb, Kb, Vb, Wb, G, cfgs, p2, c_base, cint_tab, T, out_max, out_sum, dump, dump_out):
    # cfgs[c] = the 9 widths, p2[c] = HB, XQ, EB, ST, A1B, A2B, A1, A2, bias, table row
    p = cuda.blockIdx.x * TPB + cuda.threadIdx.x
    ci = c_base + cuda.blockIdx.y
    smax = cuda.shared.array(TPB, float64)
    ssum = cuda.shared.array(TPB, float64)
    pmax = 0.0
    if p < G.shape[0] and ci < cfgs.shape[0]:
        Nc = cfgs[ci, 0]; Fq = cfgs[ci, 1]; Fk = cfgs[ci, 2]; Fv = cfgs[ci, 3]; Fw = cfgs[ci, 4]
        Ft = cfgs[ci, 5]; Pw = cfgs[ci, 6]; Fm = cfgs[ci, 7]; Fh = cfgs[ci, 8]
        cint = cint_tab[Nc]
        HB = p2[ci, 0]; XQ = p2[ci, 1]; EB = p2[ci, 2]; ST = p2[ci, 3]; A1B = p2[ci, 4]; A2B = p2[ci, 5]
        A1 = p2[ci, 6]; A2 = p2[ci, 7]; bias = p2[ci, 8]; ti = p2[ci, 9]
        K = cuda.local.array(64, int64)
        V = cuda.local.array(64, int64)
        W = cuda.local.array(16, int64)
        Hr = cuda.local.array(4, int64)
        for c in range(64):
            K[c] = conv_f(int64(Kb[p, c]), Fk, 134)
            V[c] = conv_f(int64(Vb[p, c]), Fv, 126)
        for c in range(16):
            W[c] = conv_f(int64(Wb[p, c]), Fw, 126)
        st = Fq + Fk - Ft
        tr = int64(1) << (st - 1)
        sp = Pw + Fv - Fm
        pr = int64(1) << (sp - 1)
        shh = Fh - Fm + Pw
        hmax = (int64(1) << (Fh - 1)) - 1
        fmask = (int64(1) << Ft) - 1
        F = Fh + 1 + Fw
        for i in range(16):                         # row = token i, both heads, then its 4 outputs
            for h in range(2):
                qa = conv_q(int64(Qb[p, 4 * i + 2 * h]), cint, Nc, Fq)
                qb = conv_q(int64(Qb[p, 4 * i + 2 * h + 1]), cint, Nc, Fq)
                r = int64(0); D = int64(0); M0 = int64(0); M1 = int64(0)
                for j in range(8):
                    ka = 8 * j + 2 * h
                    kb = ka + 4
                    # S1
                    t0 = (qa * K[ka] + qb * K[ka + 1] + tr) >> st
                    t1 = (qa * K[kb] + qb * K[kb + 1] + tr) >> st
                    # S2
                    f0 = t0 >> Ft
                    f1 = t1 >> Ft
                    r0 = t0 & fmask
                    r1 = t1 & fmask
                    c0 = f0
                    if r0 != 0:
                        c0 += 1
                    c1 = f1
                    if r1 != 0:
                        c1 += 1
                    rn = max(c0, c1)
                    if j > 0 and r > rn:
                        rn = r
                    w0 = pow2(r0, Ft, T, ti, HB, XQ, EB, ST, A1B, A2B, A1, A2, bias) >> min(rn - f0, 31)
                    w1 = pow2(r1, Ft, T, ti, HB, XQ, EB, ST, A1B, A2B, A1, A2, bias) >> min(rn - f1, 31)
                    # S3
                    pv0 = (w0 * V[ka] + w1 * V[kb] + pr) >> sp
                    pv1 = (w0 * V[ka + 1] + w1 * V[kb + 1] + pr) >> sp
                    if j == 0:
                        D = w0 + w1
                        M0 = pv0
                        M1 = pv1
                    else:
                        dl = min(rn - r, 31)
                        D = (D >> dl) + w0 + w1
                        M0 = (M0 >> dl) + pv0
                        M1 = (M1 >> dl) + pv1
                    r = rn
                Hr[2 * h] = hval(M0, D, shh, hmax)
                Hr[2 * h + 1] = hval(M1, D, shh, hmax)
            for o in range(4):
                acc = Hr[0] * W[4 * o] + Hr[1] * W[4 * o + 1] + Hr[2] * W[4 * o + 2] + Hr[3] * W[4 * o + 3]
                v = to_fp(acc, F)
                if dump:
                    dump_out[p, 4 * i + o] = v
                err = abs(v - G[p, 4 * i + o])
                if err > pmax:
                    pmax = err
    smax[cuda.threadIdx.x] = pmax
    ssum[cuda.threadIdx.x] = pmax
    cuda.syncthreads()
    s = TPB // 2
    while s > 0:
        if cuda.threadIdx.x < s:
            if smax[cuda.threadIdx.x + s] > smax[cuda.threadIdx.x]:
                smax[cuda.threadIdx.x] = smax[cuda.threadIdx.x + s]
            ssum[cuda.threadIdx.x] += ssum[cuda.threadIdx.x + s]
        cuda.syncthreads()
        s //= 2
    if cuda.threadIdx.x == 0 and ci < cfgs.shape[0]:
        cuda.atomic.max(out_max, ci, smax[0])
        cuda.atomic.add(out_sum, ci, ssum[0])

#=============================================================
#                           Host
#=============================================================
class Sweeper:
    # holds the patterns / golden and the ATTN_POW2 tables on the GPU
    def __init__(self, Q, K, V, W, G, pw_range, ft_range):
        self.P = G.shape[0]
        self.d = [cuda.to_device(np.ascontiguousarray(x)) for x in (Q, K, V, W, G)]
        self.cint = cuda.to_device(np.array([round(fm.C * (1 << n)) for n in range(40)], dtype=np.int64))
        self.tables = {}; self.d_T = None
        # default pow2 (64-entry table) for every (Ft, Pw) of the width sweep
        self.ft_min, self.pw_min = ft_range[0], pw_range[0]
        self.lut = np.zeros((ft_range[1] - ft_range[0] + 1, pw_range[1] - pw_range[0] + 1, 10), dtype=np.int64)
        for ft in range(ft_range[0], ft_range[1] + 1):
            for pw in range(pw_range[0], pw_range[1] + 1):
                self.lut[ft - self.ft_min, pw - self.pw_min] = self.p2_row(ft, pw, fm.pow2_defaults(pw))
        self.dummy = cuda.to_device(np.zeros((1, 64), dtype=np.float64))

    def table(self, HB, TB, Pw):
        # row of the table for (HB, TB, Pw), built on first use
        key = (HB, TB, Pw)
        if key not in self.tables:
            self.tables[key] = (len(self.tables), fm.pow2_table(HB, TB, Pw))
            self.d_T = None
        return self.tables[key][0]

    def p2_row(self, Ft, Pw, q, consts=None):
        # HB, XQ, EB, ST, A1B, A2B, A1, A2, bias, table row for a pow2 parameter set q
        A1, A2, bias = consts if consts is not None else fm.pow2_params(Ft, Pw, **q)[1:4]
        return [q['HB'], q['XQ'], q['EB'], q['ST'], q['A1B'], q['A2B'], A1, A2, bias, self.table(q['HB'], q['TB'], Pw)]

    def p2_default(self, cfgs):
        return self.lut[cfgs[:, 5].astype(np.int64) - self.ft_min, cfgs[:, 6].astype(np.int64) - self.pw_min]

    def upload(self):
        if self.d_T is None:
            T = np.zeros((len(self.tables), 256), dtype=np.int64)
            for i, t in self.tables.values(): T[i, :len(t)] = t
            self.d_T = cuda.to_device(T)

    def run(self, cfgs, p2=None, batch=1024, dump_n=0):
        # cfgs: (n, 9) widths (fm.KEYS), p2: (n, 10) pow2 rows (default pow2 when None)
        # returns (max error, mean of the per-pattern max) [, outputs when dump_n]
        if p2 is None: p2 = self.p2_default(cfgs)
        self.upload()
        d_cfg = cuda.to_device(np.ascontiguousarray(cfgs, dtype=np.int64))
        d_p2 = cuda.to_device(np.ascontiguousarray(p2, dtype=np.int64))
        n = len(cfgs)
        d_max = cuda.to_device(np.zeros(n, dtype=np.float64)); d_sum = cuda.to_device(np.zeros(n, dtype=np.float64))
        dump_out = cuda.to_device(np.zeros((self.P, 64), dtype=np.float64)) if dump_n else self.dummy
        gx = (self.P + TPB - 1) // TPB
        for c0 in range(0, n, batch):
            nb = min(batch, n - c0)
            sweep_kernel[(gx, nb), TPB](*self.d, d_cfg, d_p2, c0, self.cint, self.d_T,
                                        d_max, d_sum, 1 if dump_n else 0, dump_out)
        cuda.synchronize()
        res = d_max.copy_to_host(), d_sum.copy_to_host() / self.P
        if dump_n: return res + (dump_out.copy_to_host(),)
        return res

def sweep_all(sw, cfgs, ckpt, chunk=65536, batch=1024, log=print, save_every=300):
    # full sweep with checkpoints: finished chunks are saved to ckpt (.npz) every save_every
    # seconds and at the end, a rerun resumes from there
    n = len(cfgs)
    if os.path.exists(ckpt):
        d = np.load(ckpt); mx, mn, done = d['mx'], d['mn'], int(d['done'])
        if len(mx) != n: raise ValueError('checkpoint does not match the config list')
    else:
        mx = np.zeros(n); mn = np.zeros(n); done = 0
    t0 = time.time(); start = done; saved = t0
    while done < n:
        hi = min(done + chunk, n)
        a, b = sw.run(cfgs[done:hi], batch=batch)
        mx[done:hi] = a; mn[done:hi] = b; done = hi
        if done == n or time.time() - saved > save_every:
            np.savez(ckpt, mx=mx, mn=mn, done=done); saved = time.time()
        el = time.time() - t0; rate = (done - start) / el
        log('%d / %d configs  %.0f s  (%.1f configs/s, %.1f min left)' % (done, n, el, rate, (n - done) / rate / 60))
    return mx, mn

Overwriting fxp_gpu.py


In [4]:
# Cell 4: patterns, golden, GPU data
import fxp_model as fm
import fxp_gpu as fg
importlib.reload(fm); importlib.reload(fg)
from numba import cuda

files = fm.make_patterns(PAT_DIR, GEN, SEEDS)
pool = Pool(os.cpu_count())
t0 = time.time()
Q, K, V, W, G = fm.load_data(files, os.path.join(PAT_DIR, 'data.npz'), pool)
sw = fg.Sweeper(Q, K, V, W, G, RANGES['Pw'], RANGES['Ft'])
print('%d patterns, golden ready in %.0f s, GPU: %s' % (len(G), time.time() - t0, cuda.get_current_device().name.decode() if isinstance(cuda.get_current_device().name, bytes) else cuda.get_current_device().name))

10000 patterns, golden ready in 0 s, GPU: NVIDIA GeForce RTX 4060


In [5]:
# Cell 5: GPU kernel vs CPU bit-exact model (all 64 outputs of the first 300 patterns, bit for bit)
# the GPU kernel is a second implementation of fxp_model.model: rerun this after editing either one
hw = [CURRENT_HW[k] for k in fm.KEYS]
mx_hw, mn_hw, dump = sw.run(np.array([hw]), dump_n=1)
p_hw = dict(fm.BASE); p_hw.update(CURRENT_HW)
pw = fm.make_pow2(p_hw)
bad = sum(1 for i in range(300) for a, b in zip(fm.model(fm.as_pat(Q, K, V, W, i), p_hw, pw), dump[i]) if a != b)
print('current hardware: outputs GPU vs CPU %d mismatches, max err %.3e' % (bad, mx_hw[0]))
assert bad == 0

C:\Users\user\Desktop\fattn_gpu\venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 79 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


current hardware: outputs GPU vs CPU 0 mismatches, max err 5.895e-07


In [6]:
# Cell 6: full sweep of all width combinations (resumes from the checkpoint if interrupted)
# configs are built as int8 in cartesian order, then only FT >= PW - 9 is kept
axes = [np.arange(RANGES[k][0], RANGES[k][1] + 1, dtype=np.int8) for k in fm.KEYS]
n = int(np.prod([len(x) for x in axes]))
cfgs = np.empty((n, 9), dtype=np.int8)
inner = n
for i, x in enumerate(axes):
    inner //= len(x)
    cfgs[:, i] = np.tile(np.repeat(x, inner), n // (inner * len(x)))
cfgs = cfgs[cfgs[:, 5] >= cfgs[:, 6] - 9]
tag = '_'.join('%s%d-%d' % (k, *RANGES[k]) for k in fm.KEYS)
ckpt = os.path.join(OUT_DIR, 'sweep_%s.npz' % tag)
print('%d configs (%d skipped by FT >= PW - 9)' % (len(cfgs), n - len(cfgs)))
t0 = time.time()
last = [0.0]
def log(s):
    if time.time() - last[0] > 600: print(s, flush=True); last[0] = time.time()
mx, mn = fg.sweep_all(sw, cfgs, ckpt, log=log)
print('%d configs done, %.1f min in this run' % (len(cfgs), (time.time() - t0) / 60))

20744640 configs (432180 skipped by FT >= PW - 9)
20744640 configs done, 0.0 min in this run


In [7]:
# Cell 7: configurations that meet TOL, sorted by total bits (descending)
def table(mask):
    d = pd.DataFrame(cfgs[mask].astype(np.int32), columns=fm.KEYS)
    d['bits'] = d[fm.KEYS].sum(axis=1)
    d['max_err'] = mx[mask]
    d['mean_err'] = mn[mask]
    return d
def hw_row(d):
    i = np.flatnonzero(np.all(d[fm.KEYS].values == np.array([CURRENT_HW[k] for k in fm.KEYS]), axis=1))
    return int(i[0]) if len(i) else None
show = lambda d: display(d.assign(**{c: d[c].map('{:.3e}'.format) for c in ('max_err', 'mean_err')}))

ok = table(mx <= TOL).sort_values(['bits', 'max_err'], ascending=[False, True]).reset_index(drop=True)
ok.to_csv(os.path.join(OUT_DIR, 'pass_table.csv'), index=False, float_format='%.4e')
print('%d of %d configs meet max error <= %.0e (saved to results/pass_table.csv)' % (len(ok), len(cfgs), TOL))
r = hw_row(ok)
if r is not None: print('current hardware: row %d, bits %d' % (r, ok['bits'][r]))
print('first 20 rows (most bits)'); show(ok.head(20))
print('last 20 rows (fewest bits)'); show(ok.tail(20))

1294032 of 20744640 configs meet max error <= 1e-06 (saved to results/pass_table.csv)
current hardware: row 1206698, bits 193
first 20 rows (most bits)


,Nc,Fq,Fk,Fv,Fw,Ft,Pw,Fm,Fh,bits,max_err,mean_err
0,21,28,18,24,25,22,26,27,25,216,2.134e-07,3.367e-08
1,21,28,18,24,25,22,25,27,25,215,2.018e-07,3.813e-08
2,21,28,18,24,25,22,26,27,24,215,2.018e-07,4.041e-08
3,20,28,18,24,25,22,26,27,25,215,2.134e-07,3.367e-08
4,21,28,18,23,25,22,26,27,25,215,2.134e-07,4.304e-08
5,21,28,18,24,25,22,26,26,25,215,2.134e-07,3.645e-08
6,21,28,18,24,24,22,26,27,25,215,2.167e-07,3.987e-08
7,21,27,18,24,25,22,26,27,25,215,2.361e-07,3.896e-08
8,21,28,17,24,25,22,26,27,25,215,2.428e-07,3.446e-08
9,21,28,18,24,25,21,26,27,25,215,2.432e-07,3.504e-08


last 20 rows (fewest bits)


,Nc,Fq,Fk,Fv,Fw,Ft,Pw,Fm,Fh,bits,max_err,mean_err
1294012,15,27,17,21,21,18,23,23,21,186,9.959e-07,3.489e-07
1294013,16,27,16,21,21,18,23,23,21,186,9.959e-07,3.497e-07
1294014,15,25,16,22,20,18,25,24,21,186,9.976e-07,3.528e-07
1294015,15,26,16,22,21,18,23,24,21,186,9.996e-07,2.980e-07
1294016,15,26,18,21,21,18,23,23,21,186,9.996e-07,3.487e-07
1294017,16,26,17,21,21,18,23,23,21,186,9.996e-07,3.489e-07
1294018,17,26,16,21,21,18,23,23,21,186,9.996e-07,3.498e-07
1294019,15,25,16,21,21,19,23,23,22,185,9.355e-07,2.985e-07
1294020,15,26,16,21,21,18,23,23,22,185,9.543e-07,3.019e-07
1294021,15,25,16,21,22,18,23,23,22,185,9.584e-07,2.764e-07


In [8]:
# Cell 8: candidates for synthesis: max error <= MARGIN_TOL, sorted by estimated area (smallest first)
# est_area_k = sum(width * AREA_PER_BIT): only the width-dependent part of the area (thousands of
# area units), so only differences between rows mean something
w = np.array([AREA_PER_BIT[k] for k in fm.KEYS])
cand = table(mx <= MARGIN_TOL)
cand['est_area_k'] = cand[fm.KEYS].values @ w
cand = cand.sort_values(['est_area_k', 'max_err'], ascending=[True, True]).reset_index(drop=True)
cand.to_csv(os.path.join(OUT_DIR, 'area_table.csv'), index=False, float_format='%.4e')
print('%d configs meet max error <= %.0e (saved to results/area_table.csv)' % (len(cand), MARGIN_TOL))
r = hw_row(cand)
if r is not None: print('current hardware: row %d, est_area_k %.1f (best %.1f)' % (r, cand['est_area_k'][r], cand['est_area_k'][0]))
show(cand.head(30).assign(est_area_k=cand['est_area_k'].head(30).map('{:.1f}'.format)))

238568 configs meet max error <= 6e-07 (saved to results/area_table.csv)
current hardware: row 0, est_area_k 820.0 (best 820.0)


,Nc,Fq,Fk,Fv,Fw,Ft,Pw,Fm,Fh,bits,max_err,mean_err,est_area_k
0,19,25,16,22,22,20,23,24,22,193,5.895e-07,1.759e-07,820.0
1,20,25,16,22,22,20,23,24,22,194,5.895e-07,1.759e-07,820.1
2,21,25,16,22,22,20,23,24,22,195,5.895e-07,1.759e-07,820.2
3,19,25,16,22,22,19,23,25,22,193,5.883e-07,1.719e-07,820.5
4,20,25,16,22,22,19,23,25,22,194,5.883e-07,1.719e-07,820.6
5,21,25,16,22,22,19,23,25,22,195,5.883e-07,1.719e-07,820.7
6,19,25,16,22,22,21,23,25,22,195,5.972e-07,1.678e-07,821.5
7,20,25,16,22,22,21,23,25,22,196,5.972e-07,1.678e-07,821.6
8,21,25,16,22,22,21,23,25,22,197,5.972e-07,1.678e-07,821.7
9,19,25,16,22,22,21,23,26,22,196,5.972e-07,1.655e-07,822.5


In [9]:
# Cell 9: ATTN_POW2 internal precision with the 9 widths fixed at CURRENT_HW
# (needs cells 1 ~ 5) -> results/pow2_table.csv: max error <= MARGIN_TOL, sorted by the cost proxy
Ft, Pw = CURRENT_HW['Ft'], CURRENT_HW['Pw']
P2 = fm.P2KEYS
grid = np.array(np.meshgrid(*[np.array(list(POW2_RANGES[k])) for k in P2], indexing='ij')).reshape(len(P2), -1).T
# xq needs at least one bit (XQ > HB) and must be a part of x (XQ <= Ft), the table <= Pw bits
grid = grid[(grid[:, 3] > grid[:, 0]) & (grid[:, 3] <= Ft) & (grid[:, 1] <= Pw)]
qs = [dict(zip(P2, map(int, r))) for r in grid]
t0 = time.time()
consts = pool.map(fm.pow2_job, [(Ft, Pw, q) for q in qs], chunksize=256)
p2 = np.array([sw.p2_row(Ft, Pw, q, c) for q, c in zip(qs, consts)], dtype=np.int64)
print('%d pow2 settings, constants in %.0f s' % (len(qs), time.time() - t0))

# GPU vs CPU on a few settings (first 100 patterns, all outputs)
hw1 = np.array([[CURRENT_HW[k] for k in fm.KEYS]])
for j in np.random.RandomState(7).choice(len(qs), 3, replace=False):
    _, _, dump = sw.run(hw1, p2=p2[j:j + 1], dump_n=1)
    pq = dict(fm.BASE); pq.update(CURRENT_HW); pq.update(qs[j]); f = fm.make_pow2(pq)
    bad = sum(1 for i in range(100) for a, b in zip(fm.model(fm.as_pat(Q, K, V, W, i), pq, f), dump[i]) if a != b)
    assert bad == 0, (qs[j], bad)
print('GPU vs CPU check on 3 settings: ok')

t0 = time.time()
pmx, pmn = sw.run(np.repeat(hw1, len(qs), axis=0), p2=p2)
print('sweep %.0f s' % (time.time() - t0))

# cost proxy per pow2 instance: x*a1, xq^2, xq^2*a2, T*e array bits, table bits
def bits(v): return int(v).bit_length()
rows = []
for q, (A1, A2, bias), mx_, mn_ in zip(qs, consts, pmx, pmn):
    xb, xqb = Ft - q['HB'], q['XQ'] - q['HB']
    eb = bits(int((2.0 ** (2.0 ** -q['HB']) - 1) * 2 ** q['EB']) + 1)
    mult = xb * bits(A1) + xqb * xqb + 2 * xqb * bits(A2) * (A2 > 0) + (Pw + 1 - q['ST']) * eb
    rom = (1 << q['HB']) * (q['TB'] + 1)
    rows.append(dict(q, max_err=mx_, mean_err=mn_, mult_bits=mult, rom_bits=rom, cost=mult + ROM_W * rom))
pt = pd.DataFrame(rows)
cur = fm.pow2_defaults(Pw)
cur_row = pt[np.all(pt[P2].values == np.array([cur[k] for k in P2]), axis=1)].iloc[0]
pt = pt[pt['max_err'] <= MARGIN_TOL].sort_values(['cost', 'max_err']).reset_index(drop=True)
pt.to_csv(os.path.join(OUT_DIR, 'pow2_table.csv'), index=False, float_format='%.4e')
print('%d of %d pow2 settings keep max error <= %.0e (saved to results/pow2_table.csv)' % (len(pt), len(qs), MARGIN_TOL))
print('current pow2: max err %.3e, mult_bits %d, rom_bits %d, cost %.0f' % (cur_row['max_err'], cur_row['mult_bits'], cur_row['rom_bits'], cur_row['cost']))
display(pt.head(30).assign(max_err=pt['max_err'].head(30).map('{:.3e}'.format), mean_err=pt['mean_err'].head(30).map('{:.3e}'.format)))

360150 pow2 settings, constants in 10 s


GPU vs CPU check on 3 settings: ok


sweep 88 s


30790 of 360150 pow2 settings keep max error <= 6e-07 (saved to results/pow2_table.csv)
current pow2: max err 5.895e-07, mult_bits 702, rom_bits 1536, cost 1086


,HB,TB,A1B,XQ,A2B,EB,ST,max_err,mean_err,mult_bits,rom_bits,cost
0,5,22,13,15,7,22,8,5.852e-07,1.808e-07,667,736,851.0
1,5,22,14,15,7,22,8,5.852e-07,1.808e-07,682,736,866.0
2,5,22,13,15,7,23,8,5.852e-07,1.807e-07,683,736,867.0
3,6,22,13,13,3,22,9,5.860e-07,1.806e-07,499,1472,867.0
4,5,22,13,15,7,22,7,5.852e-07,1.792e-07,684,736,868.0
5,5,22,13,15,8,22,8,5.852e-07,1.808e-07,687,736,871.0
6,5,23,13,15,7,22,7,5.852e-07,1.789e-07,684,768,876.0
7,5,22,15,15,7,22,8,5.852e-07,1.808e-07,697,736,881.0
8,6,22,13,13,4,22,9,5.860e-07,1.806e-07,513,1472,881.0
9,6,22,14,13,3,22,9,5.860e-07,1.806e-07,513,1472,881.0
